# First prototype for HyperBubble Resolution-Oriented GNN

In [17]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Przemek\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

# reverse mapping for debug
ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    # tokens is 1D LongTensor, e.g. [21] (a single sequence)
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [19]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long (embed this; no one-hot)
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # nodes with cov
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")  # may be None
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            # node features
            seq_tokens=seq_tokens,    # Long [N, K]
            x_cov=x_cov,              # Float [N, 1]
            # graph
            edge_index=edge_index,    # Long [2, E]
            edge_attr=edge_attr,      # Float [E, 5]
            start_idx=start_idx,      # Long []
            end_idx=end_idx,          # Long []
            num_nodes=seq_tokens.size(0),
            # labels
            y_edge_mask=y_edge_mask,                      # Long [E]
            label_path_idx=torch.tensor(label_path_idx)   # Long []
        )
        # keep only tensors in Data to avoid collation issues
        # (bubble_id/k useful but optional)
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


In [23]:
# Point to one or more JSONL files from your pipeline
jsonl_paths = [
    "./sample_data.jsonl",
]

dataset = HyperbubbleDataset(jsonl_paths)

# Peek
sample = dataset[0]
print("num_nodes:", sample.num_nodes)
print("num_edges:", sample.edge_index.size(1))

for i, (tok_row, cov) in enumerate(zip(sample.seq_tokens, sample.x_cov)):
    seq = tokens_to_seq(tok_row)
    print(f"node {i}: seq={seq}, cov={cov.item()}")

# Dataloader for batching graphs
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

num_nodes: 6
num_edges: 4
node 0: seq=CGGGCGGACCATCGACTGGCT, cov=24.0
node 1: seq=CGTACCGGGCGGACCATCGAC, cov=24.0
node 2: seq=GCGGTGGGCTACCTGAATCAC, cov=25.0
node 3: seq=ATCGGGCGGTGGGCTACCTGA, cov=14.0
node 4: seq=GGCGGTGGGCTACCTGAATCA, cov=21.0
node 5: seq=GTACCGGGCGGACCATCGACT, cov=25.0


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool

# ---- Node encoder: tokens -> embedding, plus coverage ----
class NodeEncoder(nn.Module):
    def __init__(self, vocab_size=5, emb_dim=64, cov_dim=16, gru_hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)  # tokens: 0..4
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=gru_hidden,
                          batch_first=True, bidirectional=False)
        self.cov_mlp = nn.Sequential(
            nn.Linear(1, cov_dim), nn.ReLU(),
            nn.Linear(cov_dim, gru_hidden)
        )
        self.fuse = nn.Sequential(
            nn.Linear(gru_hidden * 2, gru_hidden), nn.ReLU(),
            nn.Linear(gru_hidden, gru_hidden)
        )

    def forward(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] Long
        x_cov     : [N, 1] Float
        returns   : [N, H]
        """
        x = self.emb(seq_tokens)               # [N, K, E]
        _, h = self.gru(x)                     # h: [1, N, H]
        h = h.squeeze(0)                       # [N, H]
        h_cov = self.cov_mlp(x_cov)            # [N, H]
        h = torch.cat([h, h_cov], dim=-1)      # [N, 2H]
        return self.fuse(h)                    # [N, H]


# ---- GNN that predicts which edges are on the labeled path ----
class SeqEdgeGNN(nn.Module):
    def __init__(self, node_hidden=64, edge_in_dim=4, gnn_layers=2):
        """
        edge_in_dim: we use [len_nodes, len_bp, cov_min, cov_mean] (drop on_ref)
        """
        super().__init__()
        self.enc = NodeEncoder(gru_hidden=node_hidden)

        # Edge encoder used by GINEConv:
        self.edge_nn = nn.Sequential(
            nn.Linear(edge_in_dim, node_hidden), nn.ReLU(),
            nn.Linear(node_hidden, node_hidden)
        )
        self.convs = nn.ModuleList([
            GINEConv(
                nn=nn.Sequential(
                    nn.Linear(node_hidden, node_hidden), nn.ReLU(),
                    nn.Linear(node_hidden, node_hidden)
                ),
                edge_dim=node_hidden
            ) for _ in range(gnn_layers)
        ])
        self.norms = nn.ModuleList([nn.BatchNorm1d(node_hidden) for _ in range(gnn_layers)])

        # Edge scoring head: f([h_u,h_v, edge_feat])
        self.edge_feat_mlp = nn.Sequential(
            nn.Linear(edge_in_dim, node_hidden), nn.ReLU(),
            nn.Linear(node_hidden, node_hidden)
        )
        self.edge_scorer = nn.Sequential(
            nn.Linear(node_hidden * 3, node_hidden), nn.ReLU(),
            nn.Linear(node_hidden, 1)  # logit
        )

    def forward(self, data):
        # Node encode
        x = self.enc(data.seq_tokens, data.x_cov)   # [N, H]

        # Prepare edge features (drop last column "on_ref" to avoid leakage)
        if data.edge_attr.size(0) > 0:
            e_in = data.edge_attr[:, :4]           # [E, 4]
            e_enc = self.edge_nn(e_in)             # [E, H]
        else:
            e_in = data.edge_attr
            e_enc = e_in

        # GNN layers
        h = x
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, data.edge_index, e_enc)
            h = bn(h)
            h = F.relu(h)

        # Edge logits
        u, v = data.edge_index                      # [E], [E]
        if e_in.size(0) > 0:
            ef = self.edge_feat_mlp(e_in)           # [E, H]
        else:
            ef = torch.zeros((u.numel(), h.size(-1)), device=h.device)

        edge_repr = torch.cat([h[u], h[v], ef], dim=-1)  # [E, 3H]
        logits = self.edge_scorer(edge_repr).squeeze(-1) # [E]
        return logits


# ---- Small train/eval utility on your DataLoader ----
def train_one_epoch(model, loader, device, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    total_loss, total_edges = 0.0, 0
    bce = nn.BCEWithLogitsLoss(reduction='sum')

    for data in loader:
        data = data.to(device)
        # figure out which edges are from graphs that have a label path
        # graph index per edge = graph of the source node
        edge_graph = data.batch[data.edge_index[0]]  # [E]
        has_label_graph = (data.label_path_idx >= 0) # [num_graphs]
        valid_edge = has_label_graph[edge_graph]     # [E]

        if valid_edge.sum() == 0:
            continue

        logits = model(data)
        y = data.y_edge_mask.float()

        loss = bce(logits[valid_edge], y[valid_edge])
        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        total_edges += int(valid_edge.sum())

    avg_loss = total_loss / max(1, total_edges)
    return avg_loss


@torch.no_grad()
def eval_batch(model, loader, device, max_batches=1):
    model.eval()
    for i, data in enumerate(loader):
        if i >= max_batches:
            break
        data = data.to(device)
        logits = model(data)
        pred = (logits.sigmoid() > 0.5).long()

        edge_graph = data.batch[data.edge_index[0]]
        has_label_graph = (data.label_path_idx >= 0)
        valid_edge = has_label_graph[edge_graph]
        y = data.y_edge_mask

        acc = (pred[valid_edge] == y[valid_edge]).float().mean().item() if valid_edge.any() else float('nan')

        print(f"[eval] batch {i}  E={logits.numel()}  "
              f"valid={int(valid_edge.sum())}  acc={acc:.3f}")
        # optional: show a few edges
        show = min(10, logits.numel())
        print(" first few logits:", logits[:show].detach().cpu().numpy())
        print(" first few preds :", pred[:show].detach().cpu().numpy())
        print(" first few labels:", y[:show].detach().cpu().numpy())


# ---- Wire it up ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SeqEdgeGNN(node_hidden=64, edge_in_dim=4, gnn_layers=2).to(device)

# quick sanity run
batch = next(iter(loader))
batch = batch.to(device)
with torch.no_grad():
    logits = model(batch)
print("logits shape:", logits.shape)  # [E_total_in_batch]

# tiny training loop (just to test end-to-end)
for epoch in range(3):
    loss = train_one_epoch(model, loader, device, lr=1e-3)
    print(f"epoch {epoch} | edge-averaged BCE: {loss:.4f}")

eval_batch(model, loader, device, max_batches=1)


logits shape: torch.Size([82])
epoch 0 | edge-averaged BCE: 0.6672
epoch 1 | edge-averaged BCE: 0.6508
epoch 2 | edge-averaged BCE: 0.6338
[eval] batch 0  E=82  valid=70  acc=0.586
 first few logits: [-2.724971   -0.3897437  -0.39859158 -0.92232496 -0.06700556 -0.07481256
 -0.1236309  -0.11582386 -0.07656713 -0.08437431]
 first few preds : [0 0 0 0 0 0 0 0 0 0]
 first few labels: [0 1 0 0 1 0 0 1 1 0]
